In [1]:
import pandas as pd
import numpy as np

In [2]:


df = pd.DataFrame({
    "order_id": range(1, 11),
    "cust_Name": ["Ivan", "maria", "PETR", "anna", "Sergey", "olga", "dmitry", "elena", "alex", "nina"],
    "region_Code": ["RU-MOW", "RU-SPB", "RU-MOW", "US-CA", "RU-SPB", "RU-MOW", "US-CA", "RU-SPB", "RU-MOW", "US-CA"],
    "order_Date": ["2024-01-05", "2024-02-10", "2024-01-15", "2024-03-20", "2024-02-28", "2024-01-08", "2024-03-15", "2024-02-05", "2024-01-25", "2024-03-10"],
    "Item_Price": [1500.0, 2000.0, 500.0, 3000.0, 1200.0, 800.0, 2500.0, 900.0, 1700.0, 4000.0],
    "Quantity": [2, 1, 5, 1, 3, 2, 1, 4, 2, 1],
    "Discount_Pct": [0.1, np.nan, 0.0, 0.15, 0.05, np.nan, 0.2, 0.0, 0.1, 0.25],
    "status": ["completed", "cancelled", "completed", "completed", "refunded", "completed", "pending", "completed", "completed", "cancelled"],
    "Payment_Method": ["card", "cash", "card", "crypto", "card", "cash", "card", "card", "crypto", "cash"]
})

In [11]:
new_df = df.query("status=='completed' or status=='refunded' ")\
         .rename(columns={
             'cust_Name': 'customer_name',
             'region_Code': 'region',
             'order_Date': 'order_date', 
             'Item_Price': 'price', 
             'Discount_Pct': 'discount'})\
         .astype({
              'order_date': 'datetime64[ns]',
              'discount': 'float'})\
         .assign(final_amount = lambda df: df['price'] * df['Quantity'] * (1 - df['discount']))\
         .assign(is_high_value = lambda df: df['final_amount'] > 2000)\
         .groupby(['region','is_high_value'])\
         .agg({
            'final_amount': 'sum', 
            'order_id': 'count', 
            'price': 'mean'})\
         .rename(columns={
            'final_amount': 'total_sales', 
            'order_id': 'order_count', 
            'price': 'avg_price'})\
         .sort_values('total_sales', ascending=1)\
         .sort_values('order_count', ascending=0)\
         .reset_index()       

In [12]:
new_df

,region,is_high_value,total_sales,order_count,avg_price
0,RU-MOW,True,8260.0,3,1233.333333
1,RU-SPB,True,7020.0,2,1050.000000
2,US-CA,True,2550.0,1,3000.000000
3,RU-MOW,False,0.0,1,800.000000
